# 002 · Reading and Writing Files

> **Standard Library › Files & Paths** · Python 3.14

The built-in `open()` function returns a *file object* (a stream) for reading or writing text or bytes. Combined with a `with` block it covers everything from `File.ReadAllText` to `FileStream` with manual seeking.

**In this notebook you will learn:**

- `open()` modes (`r`, `w`, `a`, `x`, `b`, `+`) and how they map to `FileMode`/`FileAccess`
- why you should always pass `encoding="utf-8"`, and how newlines are translated
- reading whole files, line by line (lazily), and writing with `write`/`writelines`
- binary files, `seek` and `tell`
- the exceptions you'll meet, and the atomic-write pattern

## .NET mental model

| .NET | Python |
|---|---|
| `File.ReadAllText(path)` | `Path(path).read_text(encoding="utf-8")` or `open(...).read()` |
| `File.ReadLines(path)` (lazy) | `for line in f:` |
| `File.ReadAllLines(path)` | `f.read().splitlines()` / `f.readlines()` |
| `new StreamReader(path)` / `StreamWriter` | `open(path, "r"/"w", encoding="utf-8")` (a `TextIOWrapper`) |
| `new FileStream(path, FileMode.Open)` | `open(path, "rb")` (a `BufferedReader`) |
| `using (...) { }` | `with open(...) as f:` |

One function, `open()`, replaces most of `File.*`, `StreamReader`, `StreamWriter` and `FileStream`. The **mode string** decides text vs binary and read vs write.

All examples use a throw-away directory *(see Standard Library › Files & Paths › 004 · Temporary Files and Directories)*:

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
print(root.is_dir())

## Writing and reading text — the basic pattern

```csharp
// C#
using (var writer = new StreamWriter("greeting.txt"))   // UTF-8 by default
{
    writer.WriteLine("Hello");
}
string text = File.ReadAllText("greeting.txt");
```

`with` closes the file when the block ends — even if an exception is raised — just like `using` *(see Syntax › Error Handling › 003 · Context Managers)*.

In [ ]:
greeting = root / "greeting.txt"

with open(greeting, "w", encoding="utf-8") as f:
    f.write("Hello\n")
    f.write("Grüße aus Python\n")

with open(greeting, encoding="utf-8") as f:    # "r" (read text) is the default mode
    text = f.read()

print(repr(text))
print(f.closed)    # True — the with block closed it

## Modes

| Mode | Meaning | .NET equivalent |
|---|---|---|
| `"r"` | read (default); file must exist | `FileMode.Open, FileAccess.Read` |
| `"w"` | write; **truncates** or creates | `FileMode.Create` |
| `"a"` | append; creates if missing | `FileMode.Append` |
| `"x"` | exclusive create; fails if it exists | `FileMode.CreateNew` |
| `"b"` | binary — add to the above: `"rb"`, `"wb"` | `FileStream` instead of `StreamReader` |
| `"+"` | read **and** write: `"r+"`, `"w+"`, `"a+"` | `FileAccess.ReadWrite` |
| `"t"` | text (the default, rarely written) | `StreamReader`/`StreamWriter` |

In [ ]:
log = root / "app.log"

with open(log, "a", encoding="utf-8") as f:   # append: creates the file
    f.write("first run\n")
with open(log, "a", encoding="utf-8") as f:   # append again: keeps old content
    f.write("second run\n")

print(log.read_text(encoding="utf-8"))

with open(log, "w", encoding="utf-8") as f:   # "w" wipes the file first!
    f.write("fresh start\n")
print(log.read_text(encoding="utf-8"))

`"x"` is the safe "create, but never overwrite" mode:

In [ ]:
try:
    with open(log, "x", encoding="utf-8") as f:
        f.write("would clobber")
except FileExistsError as e:
    print(f"{type(e).__name__}: {e.strerror}")

## Always pass `encoding="utf-8"`

⚠️ **Gotcha:** .NET's `StreamReader`/`File.ReadAllText` default to UTF-8. Python's `open()` defaults to the **locale encoding**, which can be something else (e.g. `cp1252` on some Windows setups). Code that works on your Mac may then corrupt non-ASCII text on a Windows server. Always pass `encoding="utf-8"` for text mode.

Python's "UTF-8 mode" (`python -X utf8`, or `PYTHONUTF8=1`) changes the default — and is planned to become the default in a future release — but being explicit is still the best habit. Use `encoding="utf-8-sig"` to read files that start with a BOM (as written by some Windows tools).

In [ ]:
import locale
print(locale.getencoding())   # the locale encoding (3.11+) — machine dependent!

bom_file = root / "bom.txt"
bom_file.write_bytes("﻿id,name\n".encode("utf-8"))
print(repr(bom_file.read_text(encoding="utf-8")))       # BOM shows up as '﻿'
print(repr(bom_file.read_text(encoding="utf-8-sig")))   # BOM stripped

## Reading: whole file, lines, or lazily

| Call | Returns | .NET |
|---|---|---|
| `f.read()` | the whole content as one `str` | `ReadToEnd()` |
| `f.read(n)` | up to `n` characters | `Read(buffer, 0, n)` |
| `f.readline()` | the next line, **including** `"\n"`; `""` at EOF | `ReadLine()` (but that strips the newline and returns `null` at EOF) |
| `f.readlines()` | list of all lines (with `"\n"`) | `File.ReadAllLines` |
| `for line in f:` | lines one by one, lazily | `File.ReadLines` |

In [ ]:
data = root / "sensors.txt"
data.write_text("temp,21.5\nhumidity,40\npressure,1013\n", encoding="utf-8")

with open(data, encoding="utf-8") as f:
    print(repr(f.readline()))     # 'temp,21.5\n'  — newline kept
    print(repr(f.read(8)))        # next 8 characters
    print(repr(f.readline()))     # rest of that line
    print(repr(f.readline()))     # last line
    print(repr(f.readline()))     # '' -> end of file (C# would return null)

The idiomatic way to process a file of any size is to **iterate the file object** — it streams one line at a time, like `File.ReadLines`, so memory use stays flat. Strip the trailing newline with `rstrip("\n")` (or `strip()` to drop all surrounding whitespace).

In [ ]:
readings = {}
with open(data, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        name, value = line.rstrip("\n").split(",")
        readings[name] = float(value)
        print(line_no, name, value)

print(readings)

**Tip:** for small files, `path.read_text(encoding="utf-8").splitlines()` is the one-liner equivalent of `File.ReadAllLines` — `splitlines()` removes the line endings for you.

## Writing: `write`, `writelines` and `print(file=...)`

⚠️ **Gotcha:** neither `write()` nor `writelines()` adds a newline. `writelines` is *not* the opposite of `readlines` in that sense — it just writes each string back to back (think `Write`, never `WriteLine`).

In [ ]:
out = root / "fruits.txt"
fruits = ["apple", "banana", "cherry"]

with open(out, "w", encoding="utf-8") as f:
    f.writelines(fruits)                              # oops: no separators
print(repr(out.read_text(encoding="utf-8")))

with open(out, "w", encoding="utf-8") as f:
    f.writelines(f"{name}\n" for name in fruits)      # add them yourself
print(repr(out.read_text(encoding="utf-8")))

`print(..., file=f)` behaves like `StreamWriter.WriteLine` — it adds `"\n"` and converts arguments to strings for you:

In [ ]:
with open(out, "w", encoding="utf-8") as f:
    for i, name in enumerate(fruits, start=1):
        print(i, name, sep=": ", file=f)

print(out.read_text(encoding="utf-8"))

## Newline handling

In text mode Python uses **universal newlines**:

- **Reading:** `"\r\n"`, `"\r"` and `"\n"` are all translated to `"\n"`.
- **Writing:** `"\n"` is translated to `os.linesep` (`"\r\n"` on Windows, `"\n"` elsewhere) — like `Environment.NewLine` in `WriteLine`.

Pass `newline=""` to switch translation off (needed for the `csv` module) or `newline="\n"` to force Unix endings on every OS.

In [ ]:
crlf = root / "windows.txt"
crlf.write_bytes(b"line one\r\nline two\r\n")       # a file written on Windows

with open(crlf, encoding="utf-8") as f:
    print(repr(f.read()))                           # translated to \n

with open(crlf, encoding="utf-8", newline="") as f:
    print(repr(f.read()))                           # untouched: \r\n kept

with open(root / "unix.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write("same bytes on every OS\n")
print((root / "unix.txt").read_bytes())

## Binary files

Add `b` to the mode and you get `bytes` in and out — no encoding, no newline translation. This is `FileStream` territory.

In [ ]:
blob = root / "image.bin"

with open(blob, "wb") as f:
    f.write(bytes([0x89, 0x50, 0x4E, 0x47]))     # PNG-style magic number
    f.write(bytes(range(10)))

with open(blob, "rb") as f:
    header = f.read(4)
    rest = f.read()

print(header, header == b"\x89PNG")
print(list(rest))
print(type(f).__name__)    # BufferedReader (text mode would be TextIOWrapper)

⚠️ **Gotcha:** mixing types fails loudly — text-mode files want `str`, binary files want `bytes`.

In [ ]:
try:
    with open(root / "scratch.bin", "wb") as f:   # (not blob: "wb" would truncate it first!)
        f.write("text into a binary file")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Random access: `seek` and `tell`

```csharp
// C#
using var fs = new FileStream("image.bin", FileMode.Open);
fs.Seek(4, SeekOrigin.Begin);
long pos = fs.Position;
```

`f.seek(offset, whence)` uses `0` = begin (`SeekOrigin.Begin`), `1` = current, `2` = end; `f.tell()` is `Position`.

In [ ]:
import os

with open(blob, "r+b") as f:          # read + write, no truncation
    f.seek(4)                          # skip the header
    print(f.tell(), f.read(3))         # 4 b'\x00\x01\x02'
    f.seek(-2, os.SEEK_END)            # 2 bytes before the end
    print(f.tell(), f.read())          # 12 b'\x08\t'
    f.seek(0)
    f.write(b"FAKE")                   # overwrite the first 4 bytes in place

print(blob.read_bytes()[:6])

⚠️ **Gotcha:** in **text** mode, `tell()` returns an opaque cookie, not a character index, and `seek()` only accepts `0` or a value previously returned by `tell()`. Do byte-level positioning in binary mode.

## Errors you will meet

| Python | .NET |
|---|---|
| `FileNotFoundError` | `FileNotFoundException` / `DirectoryNotFoundException` |
| `FileExistsError` (mode `"x"`) | `IOException` |
| `IsADirectoryError` | `UnauthorizedAccessException` (opening a directory) |
| `PermissionError` | `UnauthorizedAccessException` |
| `UnicodeDecodeError` | `DecoderFallbackException` (only if you opt in — .NET substitutes `�` by default) |

All of the OS ones are subclasses of `OSError`, so `except OSError` catches them all (like catching `IOException`). Exact types can differ per OS — opening a directory raises `IsADirectoryError` on macOS/Linux but `PermissionError` on Windows.

In [ ]:
cases = [
    lambda: open(root / "missing.txt", encoding="utf-8"),
    lambda: open(root, encoding="utf-8"),                            # a directory
    lambda: (root / "latin1.txt").read_text(encoding="utf-8"),
]
(root / "latin1.txt").write_bytes("café".encode("latin-1"))         # not valid UTF-8

for case in cases:
    try:
        case()
    except (OSError, UnicodeDecodeError) as e:
        print(f"{type(e).__name__}: {getattr(e, 'strerror', None) or e}")

If you'd rather substitute bad bytes than fail, use the `errors=` argument — `"replace"` gives you the .NET-like `�` behaviour *(see Standard Library › Text Processing › 004 · Unicode, Bytes and Encodings)*:

In [ ]:
print((root / "latin1.txt").read_text(encoding="utf-8", errors="replace"))

## The atomic write pattern

If a program crashes halfway through `open(path, "w")`, readers can see a truncated file. The robust pattern: write to a **temporary file in the same directory**, flush it to disk, then `os.replace()` it over the target. `os.replace` is atomic on the same file system and overwrites on every OS — the counterpart of .NET's `File.Replace` / `File.Move(src, dst, overwrite: true)`.

In [ ]:
import os
import tempfile

def atomic_write_text(path: Path, text: str) -> None:
    fd, tmp_name = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
    try:
        with open(fd, "w", encoding="utf-8") as f:   # open() accepts a file descriptor
            f.write(text)
            f.flush()
            os.fsync(f.fileno())                      # make sure bytes hit the disk
        os.replace(tmp_name, path)                    # atomic swap
    except BaseException:
        os.unlink(tmp_name)                           # never leave the temp file behind
        raise

settings = root / "settings.json"
atomic_write_text(settings, '{"theme": "dark"}\n')
atomic_write_text(settings, '{"theme": "light"}\n')
print(settings.read_text(encoding="utf-8"), end="")
print("leftover temp files:", sorted(p.name for p in root.glob(".settings*")))   # []

In [ ]:
tmp.cleanup()

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Open for reading text | `new StreamReader(path)` | `open(path, encoding="utf-8")` |
| Overwrite / create | `FileMode.Create` | `open(path, "w", encoding="utf-8")` |
| Append | `FileMode.Append` / `File.AppendAllText` | mode `"a"` |
| Create only if new | `FileMode.CreateNew` | mode `"x"` |
| Binary | `FileStream` | mode `"rb"` / `"wb"` |
| Auto-close | `using` | `with` |
| Read all | `ReadToEnd()` / `File.ReadAllText` | `f.read()` / `p.read_text()` |
| Lazy lines | `File.ReadLines` | `for line in f:` |
| Write a line | `WriteLine(s)` | `f.write(s + "\n")` / `print(s, file=f)` |
| Seek | `fs.Seek(n, SeekOrigin.Begin)` / `Position` | `f.seek(n)` / `f.tell()` |
| Atomic replace | `File.Replace` / `File.Move(..., overwrite: true)` | `os.replace(tmp, path)` |

## Exercises

1. Write the numbers 1–5, one per line, to a file. Then append the line `done`. Read the file back and print the lines without their newline characters.
2. Write a function `count_lines(path) -> int` that counts lines **lazily** (never loading the whole file into memory).
3. Given a binary file containing `b"HEADER" + bytes(range(20))`, read only bytes 10–14 (inclusive) using `seek`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    root = Path(d)

    # 1 — "w" then "a"; splitlines() drops the newlines
    nums = root / "nums.txt"
    with open(nums, "w", encoding="utf-8") as f:
        f.writelines(f"{n}\n" for n in range(1, 6))
    with open(nums, "a", encoding="utf-8") as f:
        f.write("done\n")
    print(nums.read_text(encoding="utf-8").splitlines())

    # 2 — iterate the file object; sum() consumes the generator lazily
    def count_lines(path) -> int:
        with open(path, encoding="utf-8") as f:
            return sum(1 for _ in f)

    print(count_lines(nums))   # 6

    # 3 — seek to offset 10 in binary mode, read 5 bytes
    blob = root / "data.bin"
    blob.write_bytes(b"HEADER" + bytes(range(20)))
    with open(blob, "rb") as f:
        f.seek(10)
        print(list(f.read(5)))   # [4, 5, 6, 7, 8]
```

</details>